# OAI Collector Service — Harvest and basic analytics Notebook

Author: Alessia Bardi (CNR-ISTI / OpenAIRE AMKE)

Description: this notebook calls the OAI Collector Service developed in EOSC Beyond to request the harvesting of a catalogue compliant with the EOSC guidelines for onboarding research product catalogues (the ENES Catalogue).


### Step 1: provide parameters and call the API

In [1]:
import requests
import time

# Base URL of the Harvester service
BASE_URL = "https://harvester.service.eosc-beyond.eu/oai-collector-service"

params = {
    "oaiBaseUrl": "https://catalogue.eneslab.pilot.eosc-beyond.eu/oai",
    "oaiFormat": "oai_openaire",
    "oaiSet": None,
    "oaiFrom": None,
    "oaiUntil": None,
    "max": 0,
    "notificationEmail": "PUT YOUR EMAIL TO RECEIVE NOTIFICATIONS OF COMPLETED HARVEST"
}

query = {k: v for k, v in params.items() if v is not None}

response = requests.get(f"{BASE_URL}/api/collect", params=query)
response.raise_for_status()
info = response.json()
info


{'id': 'coll-07b2f3bd-eb53-4936-8a66-52278c7f82a2',
 'oaiBaseUrl': 'https://catalogue.eneslab.pilot.eosc-beyond.eu/oai',
 'oaiFormat': 'oai_openaire',
 'oaiSet': None,
 'oaiFrom': None,
 'oaiUntil': None,
 'storageUrl': 'zip:///var/lib/michele-test-data/dumps/coll-07b2f3bd-eb53-4936-8a66-52278c7f82a2.zip',
 'publicUrl': None,
 'start': '2026-06-24T17:12:14.262755406',
 'end': None,
 'expirationDate': None,
 'max': 9223372036854775807,
 'executionStatus': 'RUNNING',
 'total': 0,
 'notificationEmail': 'PUT YOUR EMAIL TO RECEIVE NOTIFICATIONS OF COMPLETED HARVEST',
 'message': '',
 'calls': [{'url': 'https://catalogue.eneslab.pilot.eosc-beyond.eu/oai?verb=ListRecords&metadataPrefix=oai_openaire',
   'status': 'READY',
   'responseCode': 0,
   'numberOfRecords': 0}]}

### Step 2: Get the collection_id that is needed to poll for the status

In [2]:

collection_id = info.get("id")
collection_id


'coll-07b2f3bd-eb53-4936-8a66-52278c7f82a2'

### Step 3: wait for the harvesting process to finish

In [3]:

status = info.get("executionStatus")
print("Initial status:", status)

while status not in ["READY", "COMPLETED", "FAILED", "EXPIRED"]:
    time.sleep(5)
    r = requests.get(f"{BASE_URL}/api/history/{collection_id}")
    r.raise_for_status()
    info = r.json()
    status = info.get("executionStatus")
    print("Status:", status)

info


Initial status: RUNNING
Status: COMPLETED


{'id': 'coll-07b2f3bd-eb53-4936-8a66-52278c7f82a2',
 'oaiBaseUrl': 'https://catalogue.eneslab.pilot.eosc-beyond.eu/oai',
 'oaiFormat': 'oai_openaire',
 'oaiSet': None,
 'oaiFrom': None,
 'oaiUntil': None,
 'storageUrl': 'zip:///var/lib/michele-test-data/dumps/coll-07b2f3bd-eb53-4936-8a66-52278c7f82a2.zip',
 'publicUrl': 'https://harvester.service.eosc-beyond.eu/oai-collector-service/download/coll-07b2f3bd-eb53-4936-8a66-52278c7f82a2',
 'start': '2026-06-24T17:12:14.262755406',
 'end': '2026-06-24T17:12:17.220792984',
 'expirationDate': '2026-06-27T17:12:17.220792984',
 'max': 9223372036854775807,
 'executionStatus': 'COMPLETED',
 'total': 501,
 'notificationEmail': 'PUT YOUR EMAIL TO RECEIVE NOTIFICATIONS OF COMPLETED HARVEST',
 'message': '',
 'calls': [{'url': 'https://catalogue.eneslab.pilot.eosc-beyond.eu/oai?verb=ListRecords&metadataPrefix=oai_openaire',
   'status': 'COMPLETED',
   'responseCode': 200,
   'numberOfRecords': 501}]}

### Step 4: get the URL to download the harvested records and get them

In [4]:

storage_url = info.get("publicUrl")
print("Public URL:", storage_url)

zip_bytes = requests.get(storage_url).content
with open("harvest.zip", "wb") as f:
    f.write(zip_bytes)
"Downloaded: harvest.zip"


Public URL: https://harvester.service.eosc-beyond.eu/oai-collector-service/download/coll-07b2f3bd-eb53-4936-8a66-52278c7f82a2


'Downloaded: harvest.zip'

### Here the analysis of the harvested records starts

Let's look at the first one

In [5]:
import zipfile
import xml.etree.ElementTree as ET
from xml.dom import minidom

with zipfile.ZipFile("harvest.zip", "r") as z:
    names = z.namelist()
    #print("Names in zip file:", names)

    # Find first file
    xml_files = [n for n in names if n.lower().endswith(".xml") and not n.endswith("/")]

    if not xml_files:
        raise ValueError("No XML file found")

    first_xml = xml_files[0]
    print("First :", first_xml)

    xml_data = z.read(first_xml)

# Parsing
root = ET.fromstring(xml_data)

# Pretty print
xml_pretty = minidom.parseString(xml_data).toprettyxml(indent="  ")

print(xml_pretty)

First : page_0001/e6241319f3c653b69d5d13e7bbc8db47.xml
<?xml version="1.0" ?>
<record xmlns="http://www.openarchives.org/OAI/2.0/">
  
  
  <header>
    
    
    <identifier>oai:ScenarioMIP_S3:CMIP6.ScenarioMIP.CMCC.CMCC-ESM2.ssp370.r1i1p1f1.Amon.ci.gn</identifier>
    
    
    <datestamp>2025-11-17T11:24:31Z</datestamp>
    
    
    <setSpec>dataset</setSpec>
    
  
  </header>
  
  
  <metadata>
    
    
    <oaire:resource xmlns:oaire="http://namespace.openaire.eu/schema/oaire/" xmlns:datacite="http://datacite.org/schema/kernel-4" xmlns:dc="http://purl.org/dc/elements/1.1/" xmlns:rdf="http://www.w3.org/1999/02/22-rdf-syntax-ns#" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:schemaLocation="http://namespace.openaire.eu/schema/oaire/ https://www.openaire.eu/schema/repo-lit/4.0/openaire.xsd">
      
      
      <datacite:identifier identifierType="URL">https://catalogue.eneslab.pilot.eosc-beyond.eu/collections/ScenarioMIP_S3/items/CMIP6.ScenarioMIP.CMCC.CMCC-ESM2.ssp3

### Analytics: DC fields

In [6]:


import zipfile
import xml.etree.ElementTree as ET
from collections import Counter, defaultdict

NS = {
    "oai": "http://www.openarchives.org/OAI/2.0/",
    "dc": "http://purl.org/dc/elements/1.1/",
}

metadata_counter = Counter()

with zipfile.ZipFile("harvest.zip", "r") as z:
    names = z.namelist()
    xml_files = [n for n in names if n.lower().endswith(".xml") and not n.endswith("/")]

    print(f"Number of XML records: {len(xml_files)}")

    # To analyse them all: use xml_files
    # Limit to 500: use xml_files[:500]
    for xf in xml_files:
        data = z.read(xf)
        root = ET.fromstring(data)

        metadata_el = root.find(".//oai:metadata", NS)
        if metadata_el is not None:
            # Cerca tutti gli elementi DC
            for dc_el in metadata_el.findall(".//dc:*", NS):
                tag_name = dc_el.tag.split("}")[-1]  
                metadata_counter[tag_name] += 1


print("\nFrequency of DC fields:")
for tag, count in metadata_counter.most_common():
    print(f"{tag}: {count}")


Number of XML records: 501

Frequency of DC fields:
description: 1002
publisher: 501
language: 501
format: 501


### Analytics: oaire:resourceType (value, uri, resourceTypeGeneral)

In [7]:


import zipfile
import xml.etree.ElementTree as ET
from collections import Counter

NS = {
    "oai": "http://www.openarchives.org/OAI/2.0/",
    "oaire": "http://namespace.openaire.eu/schema/oaire/",
    "dc": "http://purl.org/dc/elements/1.1/",
}

resource_type_counter = Counter()
resource_type_value_counter = Counter()
resource_type_uri_counter = Counter()
resource_type_general_counter = Counter()

with zipfile.ZipFile("harvest.zip", "r") as z:
    names = z.namelist()
    xml_files = [
        n for n in names
        if n.lower().endswith(".xml") and not n.endswith("/")
    ]

    print(f"Total number of XML records: {len(xml_files)}")

    for xf in xml_files:
        data = z.read(xf)
        root = ET.fromstring(data)

        # Cerca oaire:resourceType nel blocco <metadata>
        for rtype in root.findall(".//oaire:resourceType", NS):

            value = (rtype.text or "").strip()

            uri = rtype.attrib.get("uri", None)
            rtype_general = rtype.attrib.get("resourceTypeGeneral", None)

            # Count combinations of (value, uri, resourceTypeGeneral)
            combo = (value, uri, rtype_general)
            resource_type_counter[combo] += 1

            # Separate counters
            if value:
                resource_type_value_counter[value] += 1
            if uri:
                resource_type_uri_counter[uri] += 1
            if rtype_general:
                resource_type_general_counter[rtype_general] += 1

# ======================
# RESULTS
# ======================

print("\n=== Complete combination (value, uri, resourceTypeGeneral) ===")
for combo, count in resource_type_counter.most_common():
    print(f"{combo}: {count}")

print("\n=== Values ===")
for val, count in resource_type_value_counter.most_common():
    print(f"{val}: {count}")

print("\n=== URIs ===")
for uri, count in resource_type_uri_counter.most_common():
    print(f"{uri}: {count}")

print("\n=== resourceTypeGeneral ===")
for rtg, count in resource_type_general_counter.most_common():
    print(f"{rtg}: {count}")

Total number of XML records: 501

=== Complete combination (value, uri, resourceTypeGeneral) ===
('dataset', 'http://purl.org/coar/resource_type/c_ddb1', 'dataset'): 501

=== Values ===
dataset: 501

=== URIs ===
http://purl.org/coar/resource_type/c_ddb1: 501

=== resourceTypeGeneral ===
dataset: 501
